# Taxi Trip Time Regression

This notebook predicts taxi trip duration in minutes using spatial, temporal, and route-level features.

The final modelling pipeline uses:

- geographic distance features
- rush-hour and weekend indicators
- pickup, drop-off, and route identifiers
- leakage-safe out-of-fold target encoding
- LightGBM with MAE optimisation
- early stopping and multi-seed ensembling

> The full training dataset contains approximately 2.14 million rows, so full training can take a substantial amount of time on CPU.

## 1. Project configuration

Set `DEBUG_MODE = True` to test the complete pipeline on a smaller sample.  
Set it to `False` for full training.

Expected files:

```text
data/
├── train.csv
├── test.csv
└── sample_submission.csv   # optional
```

In [ ]:
from pathlib import Path
import warnings
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

DEBUG_MODE = True
DEBUG_ROWS = 200_000
RANDOM_STATE = 42

RUSH_HOURS = [7, 8, 9, 16, 17, 18, 19]
SEEDS = [42, 7, 2024]

BASE_FEATURES = [
    "distance_km",
    "log_dist",
    "hav_km",
    "hour",
    "weekday",
    "is_rush",
    "is_weekend",
    "pickup_lat",
    "pickup_long",
    "dropoff_lat",
    "dropoff_long",
]

TARGET_ENCODING_CONFIG = [
    ("puk", 40),
    ("dok", 40),
    ("route", 30),
]

LGB_PARAMS = {
    "objective": "regression_l1",
    "metric": "mae",
    "learning_rate": 0.06,
    "num_leaves": 127,
    "min_child_samples": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 2.0,
    "force_row_wise": True,
    "verbose": -1,
}

## 2. Load the data

The target column is `trip_time`, measured in minutes.

The notebook uses relative paths so it can run after the repository is cloned to another computer.

In [ ]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(
        "Place train.csv and test.csv inside the data/ directory before running this notebook."
    )

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Original train shape:", train.shape)
print("Original test shape:", test.shape)

if DEBUG_MODE:
    train = train.sample(
        n=min(DEBUG_ROWS, len(train)),
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

    test = test.sample(
        n=min(max(DEBUG_ROWS // 10, 10_000), len(test)),
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

    print("Debug train shape:", train.shape)
    print("Debug test shape:", test.shape)

train.head()

## 3. Data quality checks

This section checks:

- data types
- missing values
- duplicate rows
- target distribution
- basic descriptive statistics

In [ ]:
print("Training columns:")
print(train.columns.tolist())

required_train_columns = {
    "pickup_lat", "pickup_long", "dropoff_lat", "dropoff_long",
    "distance_km", "hour", "weekday", "trip_time"
}
required_test_columns = required_train_columns - {"trip_time"}

missing_train = required_train_columns - set(train.columns)
missing_test = required_test_columns - set(test.columns)

if missing_train:
    raise ValueError(f"Missing training columns: {sorted(missing_train)}")
if missing_test:
    raise ValueError(f"Missing test columns: {sorted(missing_test)}")

quality_summary = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "missing": train.isna().sum(),
    "missing_pct": (train.isna().mean() * 100).round(3),
    "unique": train.nunique(dropna=False),
})

quality_summary

In [ ]:
print("Duplicate training rows:", train.duplicated().sum())
print("Duplicate test rows:", test.duplicated().sum())

train["trip_time"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

## 4. Exploratory data analysis

The plots below examine the target distribution and the relationship between trip distance and trip time.

For speed, large scatterplots use a random sample.

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(train["trip_time"].dropna(), bins=60)
plt.xlabel("Trip time (minutes)")
plt.ylabel("Frequency")
plt.title("Distribution of Taxi Trip Time")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "trip_time_distribution.png", dpi=150)
plt.show()

In [ ]:
eda_sample = train.sample(
    n=min(50_000, len(train)),
    random_state=RANDOM_STATE
)

plt.figure(figsize=(8, 6))
plt.scatter(
    eda_sample["distance_km"],
    eda_sample["trip_time"],
    s=8,
    alpha=0.25
)
plt.xlabel("Recorded distance (km)")
plt.ylabel("Trip time (minutes)")
plt.title("Distance versus Trip Time")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "distance_vs_trip_time.png", dpi=150)
plt.show()

print(
    "Pearson correlation:",
    round(eda_sample["distance_km"].corr(eda_sample["trip_time"]), 4)
)

In [ ]:
hour_summary = (
    train.groupby("hour", as_index=False)["trip_time"]
    .agg(["mean", "median", "count"])
    .reset_index()
)

hour_summary

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(hour_summary["hour"], hour_summary["median"], marker="o")
plt.xticks(range(0, 24))
plt.xlabel("Pickup hour")
plt.ylabel("Median trip time (minutes)")
plt.title("Median Trip Time by Hour")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "median_trip_time_by_hour.png", dpi=150)
plt.show()

## 5. Geographic and temporal feature engineering

The recorded trip distance is supplemented with Haversine distance, which estimates the great-circle distance between pickup and drop-off coordinates.

Additional features identify:

- log-transformed distance
- rush-hour trips
- weekend trips
- rounded pickup and drop-off zones
- pickup-to-drop-off route identifiers

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """Calculate great-circle distance between two latitude/longitude points in km."""
    earth_radius_km = 6371.0
    p = np.pi / 180.0

    a = (
        0.5
        - np.cos((lat2 - lat1) * p) / 2
        + np.cos(lat1 * p)
        * np.cos(lat2 * p)
        * (1 - np.cos((lon2 - lon1) * p))
        / 2
    )

    return 2 * earth_radius_km * np.arcsin(
        np.sqrt(np.clip(a, 0, 1))
    )


def add_features(df):
    """Create geographic, temporal, and route-level features."""
    df = df.copy()

    df["hav_km"] = haversine(
        df["pickup_lat"],
        df["pickup_long"],
        df["dropoff_lat"],
        df["dropoff_long"],
    )

    df["log_dist"] = np.log1p(df["distance_km"].clip(lower=0))
    df["is_rush"] = df["hour"].isin(RUSH_HOURS).astype("int8")
    df["is_weekend"] = (df["weekday"] >= 5).astype("int8")

    df["puk"] = (
        df["pickup_lat"].round(4).astype(str)
        + "_"
        + df["pickup_long"].round(4).astype(str)
    )

    df["dok"] = (
        df["dropoff_lat"].round(4).astype(str)
        + "_"
        + df["dropoff_long"].round(4).astype(str)
    )

    df["route"] = df["puk"] + "|" + df["dok"]

    return df


train_fe = add_features(train)
test_fe = add_features(test)

train_fe[BASE_FEATURES + ["puk", "dok", "route"]].head()

## 6. Leakage-safe target encoding

Pickup zones, drop-off zones, and routes may contain useful information about typical travel time.

Directly encoding each category using the full training target would leak target information into the training rows. This notebook therefore creates **out-of-fold target encodings**:

1. split the training set into five folds
2. build category statistics from four folds
3. encode the held-out fold
4. repeat until every training row has an encoding generated without its own target

Smoothing pulls rare categories toward the global target mean.

In [ ]:
def smooth_encoding(keys, target, smoothing):
    """Return a smoothed mean-target mapping and the global prior."""
    prior = target.mean()

    stats = (
        pd.DataFrame({"key": keys.values, "target": target.values})
        .groupby("key")["target"]
        .agg(["mean", "count"])
    )

    encoded = (
        stats["mean"] * stats["count"] + prior * smoothing
    ) / (stats["count"] + smoothing)

    return encoded, prior


def build_feature_matrices(train_df, test_df, target):
    """Build numeric LightGBM matrices with out-of-fold target encodings."""
    X_train = train_df[BASE_FEATURES].copy()
    X_test = test_df[BASE_FEATURES].copy()

    folds = KFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    for column, smoothing in TARGET_ENCODING_CONFIG:
        oof_values = np.zeros(len(train_df), dtype=np.float32)

        for train_idx, valid_idx in folds.split(train_df):
            mapping, prior = smooth_encoding(
                train_df[column].iloc[train_idx],
                target.iloc[train_idx],
                smoothing,
            )

            oof_values[valid_idx] = (
                train_df[column]
                .iloc[valid_idx]
                .map(mapping)
                .fillna(prior)
                .to_numpy()
            )

        X_train[f"te_{column}"] = oof_values

        full_mapping, full_prior = smooth_encoding(
            train_df[column],
            target,
            smoothing,
        )

        X_test[f"te_{column}"] = (
            test_df[column]
            .map(full_mapping)
            .fillna(full_prior)
            .to_numpy()
        )

    return X_train, X_test


y = train_fe["trip_time"].astype(float)
X, X_test = build_feature_matrices(train_fe, test_fe, y)

print("Training matrix shape:", X.shape)
print("Test matrix shape:", X_test.shape)
X.head()

### Optional row-order trend feature

The original competition script included a feature that increased from 0 to 1 over the training rows and was set to 1 for all test rows.

That feature is **disabled by default** because it is only defensible when row order has a genuine chronological meaning. Enable it only after verifying the dataset ordering and validating that the feature generalises.

In [ ]:
USE_ROW_ORDER_FEATURE = False

if USE_ROW_ORDER_FEATURE:
    X["row_order"] = np.linspace(0, 1, len(X))
    X_test["row_order"] = 1.0

## 7. Validation strategy

The original solution used the final 10% of rows as validation. This can be useful if row order approximates time, but it should be interpreted carefully.

The notebook reports validation MAE explicitly. Lower MAE is better.

In [ ]:
validation_cut = int(len(X) * 0.90)

X_train = X.iloc[:validation_cut]
X_valid = X.iloc[validation_cut:]
y_train = y.iloc[:validation_cut]
y_valid = y.iloc[validation_cut:]

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))

## 8. Train the validation model

Early stopping selects the number of boosting rounds that minimises validation MAE.

In [ ]:
start_time = time.time()

validation_model = lgb.train(
    params={**LGB_PARAMS, "seed": RANDOM_STATE},
    train_set=lgb.Dataset(X_train, label=y_train),
    num_boost_round=4000,
    valid_sets=[lgb.Dataset(X_valid, label=y_valid)],
    valid_names=["validation"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=120),
        lgb.log_evaluation(period=200),
    ],
)

valid_predictions = validation_model.predict(
    X_valid,
    num_iteration=validation_model.best_iteration
)

validation_mae = mean_absolute_error(y_valid, valid_predictions)

print(f"Best iteration: {validation_model.best_iteration}")
print(f"Validation MAE: {validation_mae:.5f}")
print(f"Training time: {(time.time() - start_time) / 60:.2f} minutes")

## 9. Error analysis

The following plots show prediction quality and the distribution of absolute errors.

In [ ]:
error_sample_size = min(25_000, len(y_valid))
error_rng = np.random.default_rng(RANDOM_STATE)
error_idx = error_rng.choice(
    len(y_valid),
    size=error_sample_size,
    replace=False
)

actual_sample = y_valid.iloc[error_idx].to_numpy()
pred_sample = valid_predictions[error_idx]

plt.figure(figsize=(7, 7))
plt.scatter(actual_sample, pred_sample, s=8, alpha=0.25)

plot_limit = max(
    np.percentile(actual_sample, 99),
    np.percentile(pred_sample, 99)
)
plt.plot([0, plot_limit], [0, plot_limit], linestyle="--")

plt.xlim(0, plot_limit)
plt.ylim(0, plot_limit)
plt.xlabel("Actual trip time")
plt.ylabel("Predicted trip time")
plt.title("Actual versus Predicted Trip Time")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "actual_vs_predicted.png", dpi=150)
plt.show()

In [ ]:
absolute_errors = np.abs(y_valid.to_numpy() - valid_predictions)

plt.figure(figsize=(9, 5))
plt.hist(absolute_errors, bins=60)
plt.xlabel("Absolute error (minutes)")
plt.ylabel("Frequency")
plt.title("Validation Absolute Error Distribution")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "absolute_error_distribution.png", dpi=150)
plt.show()

pd.Series(absolute_errors).describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

## 10. Feature importance

Gain importance measures how much each feature improves the model's objective across tree splits.

In [ ]:
feature_importance = (
    pd.DataFrame({
        "feature": validation_model.feature_name(),
        "importance": validation_model.feature_importance(
            importance_type="gain"
        ),
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance

In [ ]:
top_features = feature_importance.head(15).sort_values("importance")

plt.figure(figsize=(9, 6))
plt.barh(top_features["feature"], top_features["importance"])
plt.xlabel("Gain importance")
plt.title("Top LightGBM Features")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "feature_importance.png", dpi=150)
plt.show()

## 11. Train final models and create an ensemble

The final number of rounds is scaled from the 90% training subset to the full training set.

Three LightGBM models are trained with different random seeds, and their test predictions are averaged.

In [ ]:
best_iteration = validation_model.best_iteration or 1000
final_rounds = max(1, int(best_iteration / 0.90))

print("Final boosting rounds:", final_rounds)

test_predictions = np.zeros(len(X_test), dtype=np.float64)

for seed in SEEDS:
    print(f"Training final model with seed {seed}...")

    final_model = lgb.train(
        params={**LGB_PARAMS, "seed": seed},
        train_set=lgb.Dataset(X, label=y),
        num_boost_round=final_rounds,
    )

    test_predictions += final_model.predict(X_test) / len(SEEDS)

test_predictions = np.clip(test_predictions, 0, 96)

pd.Series(test_predictions).describe()

## 12. Generate the submission

The competition submission requires two columns:

- `id`
- `trip_time`

In [ ]:
if "id" not in test_fe.columns:
    raise ValueError("The test dataset must contain an 'id' column.")

submission = pd.DataFrame({
    "id": test_fe["id"],
    "trip_time": test_predictions,
})

submission_path = Path("submission.csv")
submission.to_csv(submission_path, index=False)

print(f"Saved submission to: {submission_path.resolve()}")
submission.head()

## 13. Conclusions

This pipeline combines low-level geographic information with learned route-level statistics.

Key ideas:

- Haversine distance supplements the provided distance feature.
- Rush-hour and weekend indicators capture temporal congestion patterns.
- Rounded coordinates form reusable pickup and drop-off zones.
- Out-of-fold target encoding adds route information without directly leaking each row's target.
- LightGBM efficiently handles the large dataset and nonlinear interactions.
- Multi-seed averaging reduces prediction variance.

### Limitations

- The validation split assumes that row order is meaningful.
- Rounded coordinate zones may be too coarse or too fine for some areas.
- Target encoding quality is limited for rare or unseen routes.
- The dataset does not include direct traffic, weather, or road-network information.
- Full training is computationally expensive on CPU.

### Possible improvements

- compare ordered and random validation strategies
- add cyclical hour and weekday features
- test alternative geographic grid sizes
- add route bearing and coordinate differences
- tune LightGBM with cross-validation
- blend LightGBM and CatBoost predictions